In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from pycox.models import CoxPH
from lifelines.utils import concordance_index
from model import RFG_DeepSurv

In [ ]:
## we need to run the RFG_DeepSurv Model

net = RFG_DeepSurv(in_features=X_train.shape[1], hidden_nodes=[64, 32], dropout=0.2)
optimizer = optim.SGD(net.parameters(), lr=0.0004, momentum=0.9, weight_decay=1e-4)
deepsurv_model = CoxPH(net, optimizer)

In [ ]:
## Model weight

best_model_path = "/scratch/home/subrata001/BitsLab/Brain_Age_Analysis/Results/ADNI_OASIS_DeepSurv/deepsurv3_best_model_weights_clinical_embedded.pt"
deepsurv_model.load_model_weights(str(best_model_path))

In [ ]:
## evaluate_deepsurv

def evaluate_deepsurv(model, X, durations, events, dataset_name, out_dir):
    surv = model.predict_surv_df(X)

    ev = EvalSurv(
        surv,
        durations,
        events,
        censor_surv="km"
    )

    c_index = ev.concordance_td("antolini")

    # Integrated Brier Score
    max_time = min(float(np.max(durations)), float(surv.index.max()))
    min_time = max(float(np.min(durations)), float(surv.index.min()))

    time_grid = np.linspace(min_time, max_time, 100)

    try:
        ibs = ev.integrated_brier_score(time_grid)
    except Exception as e:
        print(f"IBS failed for {dataset_name}:", e)
        ibs = np.nan

    # Brier at 3 and 5 years
    brier_3 = np.nan
    brier_5 = np.nan

    for t in [3, 5]:
        try:
            if t <= max_time:
                bs = ev.brier_score(np.array([t]))
                if t == 3:
                    brier_3 = float(bs.iloc[0])
                if t == 5:
                    brier_5 = float(bs.iloc[0])

        except Exception as e:
            print(f"{t}-year Brier failed for {dataset_name}:", e)

    results = {
        "Model": "DeepSurv-3",
        "Features": "Clinical + BrainAGE + all MRI embeddings",
        "Test cohort": dataset_name,
        "N": len(durations),
        "Events": int(np.sum(events)),
        "Censored": int(len(events) - np.sum(events)),
        "C-index": c_index,
        "IBS": ibs,
        "3-year Brier": brier_3,
        "5-year Brier": brier_5,
    }

    pd.DataFrame([results]).to_csv(
        out_dir / f"DeepSurv3_{dataset_name}_performance.csv",
        index=False
    )

    return surv, results

In [ ]:
## this X_external is coming from RFG_DeepSurv folder, data.py and training.py that we proprocessed

surv_external, results_external = evaluate_deepsurv(
    deepsurv_model,
    X_external,
    durations_external,
    events_external,
    "OASIS3_external_test",
    model_dir
)

In [ ]:
def plot_single_subject_curve(
    surv,
    subject_index,
    #title,
    #save_path,
    max_year=5
):
    single_surv = surv.iloc[:, subject_index]

    # Limit to clinically meaningful follow-up window
    single_surv = single_surv[single_surv.index <= max_year]

    plt.figure(figsize=(7, 5))
    plt.plot(single_surv.index, single_surv.values, linewidth=3, )

    for t in [3, 5, 10]:
        if t <= single_surv.index.max():
            selected_time = single_surv.index[single_surv.index >= t][0]
            surv_prob = float(single_surv.loc[selected_time])
            risk_prob = 1 - surv_prob

            plt.scatter(selected_time, surv_prob, s=100)
            plt.annotate(
            f"{t}-yr = {surv_prob:.2f}",
            xy=(selected_time, surv_prob),
            xytext=(0, 7),
            textcoords="offset points",
            ha="left",
            va="bottom",
            fontsize=14
            )

    plt.xlabel("Follow-up Time (years)", fontsize=14)
    plt.ylabel("Predicted Event-free Survival Probability", fontsize=14)
    plt.tick_params(axis="both", labelsize=14)
    # plt.title(title)
    plt.ylim(0, 1.05)
    plt.xlim(0, max_year)
    plt.tight_layout()
    plt.show()

In [1]:
# single patient data from dataframe. We can also load single patient MRI image and preprocess accordingly for model input
#Single MRI image can be proprocessed in similary way that was performed for whole dataset.

In [ ]:
plot_single_subject_curve_limited(
    surv_external,
    subject_index=0, 
    max_year=15
    # title="Single OASIS3 Subject: DeepSurv-Predicted AD-Free Survival Curve",
    # save_path= "/scratch/home/subrata001/BitsLab/Brain_Age_Analysis/Results/ADNI_OASIS_DeepSurv/single_subject_probability/oasis3_predicted_survival_curve_OAS30576.png"
)

In [ ]:
## Load and prepare saved preprocessing (clinical and mri embedding with PCA)

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import torch


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------------------------------------
# Load complete preprocessing bundle
# ------------------------------------------------------------

preprocessing_bundle = joblib.load(model_dir / "deepsurv_preprocessing_bundle_PCA20.joblib")

clinical_scaler_sklearn = preprocessing_bundle[
    "clinical_scaler"
]

embedding_scaler_sklearn = preprocessing_bundle[
    "embedding_scaler"
]

pca_sklearn = preprocessing_bundle[
    "pca"
]

clinical_cols = preprocessing_bundle[
    "clinical_cols"
]

embedding_cols = preprocessing_bundle[
    "embedding_cols"
]

pca_cols = preprocessing_bundle[
    "pca_cols"
]

final_feature_cols = preprocessing_bundle[
    "final_feature_cols"
]


print("Clinical columns:")
print(clinical_cols)

print("\nNumber of original MRI embedding features:")
print(len(embedding_cols))

print("\nPCA columns:")
print(pca_cols)

print("\nFinal DeepSurv feature order:")
print(final_feature_cols)

print("\nFinal DeepSurv input dimension:")
print(len(final_feature_cols))